In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import matplotlib.pyplot as plt

from aiconfigurator.sdk import common
from aiconfigurator.sdk.perf_database import get_database

from aiconfigurator.sdk.common import DatabaseMode

system = "h200_sxm"
database = get_database(system=system, backend="trtllm", version="1.2.0rc5")

In [ ]:
def visualize_gemm(database):
    # gemm
    n_k = [[4096, 4096], [8192, 8192], [8192, 1024], [1024, 8192]]
    m_list = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_k), figsize=(5 * len(n_k), 5 * 2))
    for i, (n, k) in enumerate(n_k):
        for color_id, quant_mode in enumerate(database._gemm_data.keys()):
            sol_math_list = []
            sol_mem_list = []
            for m in m_list:
                sol_time, sol_math, sol_mem = database.query_gemm(
                    m=m, n=n, k=k, quant_mode=quant_mode, database_mode=DatabaseMode.SOL_FULL
                )
                db_time = database.query_gemm(
                    m=m, n=n, k=k, quant_mode=quant_mode, database_mode=DatabaseMode.SILICON
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(
                m_list, sol_math_list, color=color_list[color_id], label=f"{quant_mode} math"
            )
            ax[1, i].plot(
                m_list,
                sol_mem_list,
                color=color_list[color_id],
                linestyle="--",
                label=f"{quant_mode} mem",
            )
        ax[0, i].set_title(f"n={n}, k={k}")
        ax[0, i].set_xlabel("num_tokens")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("num_tokens")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_gemm(database)

In [ ]:
def visualize_context_attention(database):
    b = 1
    n = 32
    n_kv_list = [1, 2, 4, 8, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 524288]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_kv_list), figsize=(5 * len(n_kv_list), 5 * 2))
    color_id = 0
    for i, n_kv in enumerate(n_kv_list):
        for quant_mode in database._context_attention_data.keys():
            for kvcache_quant_mode in database._context_attention_data[quant_mode].keys():
                sol_math_list = []
                sol_mem_list = []
                for s in s_list:
                    sol_time, sol_math, sol_mem = database.query_context_attention(
                        b=b,
                        s=s,
                        n=n,
                        n_kv=n_kv,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SOL_FULL,
                        prefix=0,
                    )
                    db_time = database.query_context_attention(
                        b=b,
                        s=s,
                        n=n,
                        n_kv=n_kv,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SILICON,
                        prefix=0,
                    )
                    percentage_of_math = sol_math / db_time
                    percentage_of_mem = sol_mem / db_time
                    sol_math_list.append(percentage_of_math)
                    sol_mem_list.append(percentage_of_mem)
                ax[0, i].plot(
                    s_list,
                    sol_math_list,
                    color=color_list[color_id % len(color_list)],
                    label=f"{quant_mode}_{kvcache_quant_mode} math",
                )
                ax[1, i].plot(
                    s_list,
                    sol_mem_list,
                    color=color_list[color_id % len(color_list)],
                    linestyle="--",
                    label=f"{quant_mode}_{kvcache_quant_mode} mem",
                )
                color_id += 1
        ax[0, i].set_title(f"b={b} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.show()


visualize_context_attention(database)

In [ ]:
def visualize_context_attention_with_prefix(database):
    b = 1
    n = 32
    n_kv_list = [1, 2, 4, 8, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 524288]
    prefix_factor = 0.3

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_kv_list), figsize=(5 * len(n_kv_list), 5 * 2))
    color_id = 0
    for i, n_kv in enumerate(n_kv_list):
        for quant_mode in database._context_attention_data.keys():
            for kvcache_quant_mode in database._context_attention_data[quant_mode].keys():
                sol_math_list = []
                sol_mem_list = []
                for s in s_list:
                    prefix_len = int(s * prefix_factor)
                    sol_time, sol_math, sol_mem = database.query_context_attention(
                        b=b,
                        s=s-prefix_len,
                        n=n,
                        n_kv=n_kv,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SOL_FULL,
                        prefix=prefix_len,
                    )
                    db_time = database.query_context_attention(
                        b=b,
                        s=s-prefix_len,
                        n=n,
                        n_kv=n_kv,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SILICON,
                        prefix=prefix_len,
                    )
                    percentage_of_math = sol_math / db_time
                    percentage_of_mem = sol_mem / db_time
                    sol_math_list.append(percentage_of_math)
                    sol_mem_list.append(percentage_of_mem)
                ax[0, i].plot(
                    s_list,
                    sol_math_list,
                    color=color_list[color_id % len(color_list)],
                    label=f"{quant_mode}_{kvcache_quant_mode} math",
                )
                ax[1, i].plot(
                    s_list,
                    sol_mem_list,
                    color=color_list[color_id % len(color_list)],
                    linestyle="--",
                    label=f"{quant_mode}_{kvcache_quant_mode} mem",
                )
                color_id += 1
        ax[0, i].set_title(f"b={b} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_context_attention(database)

In [ ]:
def visualize_generation_attention(database):
    b = 64
    n = 32
    n_kv_list = [1, 2, 4, 8]  # mha uses sol in current version
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_kv_list), figsize=(5 * len(n_kv_list), 5 * 2))
    for i, n_kv in enumerate(n_kv_list):
        for color_id, kvcache_quant_mode in enumerate(database._generation_attention_data.keys()):
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                sol_time, sol_math, sol_mem = database.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SOL_FULL,
                )
                db_time = database.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SILICON,
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(
                s_list,
                sol_math_list,
                color=color_list[color_id],
                label=f"{kvcache_quant_mode} math",
            )
            ax[1, i].plot(
                s_list,
                sol_mem_list,
                color=color_list[color_id],
                linestyle="--",
                label=f"{kvcache_quant_mode} mem",
            )
        ax[0, i].set_title(f"b={b} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_generation_attention(database)
# visualize_generation_attention(database_fixed)

In [ ]:
def visualize_generation_attention_b(database):
    b_list = [1, 4, 16, 64, 256, 1024]
    n = 8
    n_kv = 2
    s_list = [128, 512, 1024, 4096, 8192, 16384, 32768, 65535]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(
        2,
        len(database._generation_attention_data.keys()),
        figsize=(5 * len(database._generation_attention_data.keys()), 5 * 2),
    )
    for i, kvcache_quant_mode in enumerate(database._generation_attention_data.keys()):
        for color_id, b in enumerate(b_list):
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                sol_time, sol_math, sol_mem = database.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SOL_FULL,
                )
                db_time = database.query_generation_attention(
                    b=b,
                    s=s,
                    n=n,
                    n_kv=n_kv,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SILICON,
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(s_list, sol_math_list, color=color_list[color_id], label=f"b{b} math")
            ax[1, i].plot(
                s_list, sol_mem_list, color=color_list[color_id], linestyle="--", label=f"b{b} mem"
            )
        ax[0, i].set_title(f"kvcache={kvcache_quant_mode} n={n}, n_kv={n_kv}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_generation_attention_b(database)

In [ ]:
def visualize_context_mla_with_prefix(database):
    b = 8
    n_list = [2, 4, 8, 16, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
    prefix_scale = 0.3

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_list), figsize=(5 * len(n_list), 5 * 2))
    color_id = 0
    for i, n_q in enumerate(n_list):
        for quant_mode in database._context_mla_data.keys():
            for kvcache_quant_mode in database._context_mla_data[quant_mode].keys():
                sol_math_list = []
                sol_mem_list = []
                for s in s_list:
                    prefix_len = int(s * prefix_scale)
                    sol_time, sol_math, sol_mem = database.query_context_mla(
                        b=b,
                        s=s-prefix_len,
                        num_heads=n_q,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SOL_FULL,
                        prefix=prefix_len,
                    )
                    db_time = database.query_context_mla(
                        b=b,
                        s=s-prefix_len,
                        num_heads=n_q,
                        kvcache_quant_mode=kvcache_quant_mode,
                        fmha_quant_mode=quant_mode,
                        database_mode=DatabaseMode.SILICON,
                        prefix=prefix_len,
                    )
                    percentage_of_math = sol_math / db_time
                    percentage_of_mem = sol_mem / db_time
                    sol_math_list.append(percentage_of_math)
                    sol_mem_list.append(percentage_of_mem)
                ax[0, i].plot(
                    s_list,
                    sol_math_list,
                    color=color_list[color_id % len(color_list)],
                    label=f"{quant_mode}_{kvcache_quant_mode} math",
                )
                ax[1, i].plot(
                    s_list,
                    sol_mem_list,
                    color=color_list[color_id % len(color_list)],
                    linestyle="--",
                    label=f"{quant_mode}_{kvcache_quant_mode} mem",
                )
                color_id += 1
        ax[0, i].set_title(f"b={b} n_q={n_q}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        ax[1, i].set_ylim(0, 1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_context_mla_with_prefix(database)

In [ ]:
def visualize_generation_mla(database):
    b = 64
    n_list = [2, 4, 8, 16, 32]
    s_list = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2, len(n_list), figsize=(5 * len(n_list), 5 * 2))
    for i, n_q_per_gpu in enumerate(n_list):
        for color_id, kvcache_quant_mode in enumerate(database._generation_mla_data.keys()):
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                sol_time, sol_math, sol_mem = database.query_generation_mla(
                    b=b,
                    s=s,
                    num_heads=n_q_per_gpu,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SOL_FULL,
                )
                db_time = database.query_generation_mla(
                    b=b,
                    s=s,
                    num_heads=n_q_per_gpu,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SILICON,
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(
                s_list,
                sol_math_list,
                color=color_list[color_id],
                label=f"{kvcache_quant_mode} math",
            )
            ax[1, i].plot(
                s_list,
                sol_mem_list,
                color=color_list[color_id],
                linestyle="--",
                label=f"{kvcache_quant_mode} mem",
            )
        ax[0, i].set_title(f"b={b} n_q_per_gpu={n_q_per_gpu}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_generation_mla(database)
# visualize_generation_attention(database_fixed)

In [ ]:
def visualize_generation_mla_b(database):
    b_list = [1, 4, 16, 64, 256, 1024]
    num_q = 16
    s_list = [128, 512, 1024, 4096, 8192, 16384, 32768]

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(
        2,
        len(database._generation_mla_data.keys()),
        figsize=(5 * len(database._generation_mla_data.keys()), 5 * 2),
    )
    for i, kvcache_quant_mode in enumerate(database._generation_mla_data.keys()):
        for color_id, b in enumerate(b_list):
            sol_math_list = []
            sol_mem_list = []
            for s in s_list:
                sol_time, sol_math, sol_mem = database.query_generation_mla(
                    b=b,
                    s=s,
                    num_heads=num_q,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SOL_FULL,
                )
                db_time = database.query_generation_mla(
                    b=b,
                    s=s,
                    num_heads=num_q,
                    kvcache_quant_mode=kvcache_quant_mode,
                    database_mode=DatabaseMode.SILICON,
                )
                percentage_of_math = sol_math / db_time
                percentage_of_mem = sol_mem / db_time
                sol_math_list.append(percentage_of_math)
                sol_mem_list.append(percentage_of_mem)
            ax[0, i].plot(s_list, sol_math_list, color=color_list[color_id], label=f"b{b} math")
            ax[1, i].plot(
                s_list, sol_mem_list, color=color_list[color_id], linestyle="--", label=f"b{b} mem"
            )
        ax[0, i].set_title(f"kvcache={kvcache_quant_mode} n_q_per_gpu={num_q}")
        ax[0, i].set_xlabel("s")
        ax[0, i].set_ylabel("math sol %")
        ax[0, i].set_ylim(0, 1)
        ax[0, i].legend()
        ax[1, i].set_xlabel("s")
        ax[1, i].set_ylabel("mem sol %")
        # ax[1,i].set_ylim(0,1)
        ax[1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_generation_mla_b(database)

In [ ]:
def visualize_dsa_module(database):
    b = 1
    num_heads = 128
    index_n_heads = 64
    index_head_dim = 128
    index_topk = 2048
    kv_cache_dtype = common.KVCacheQuantMode.float16
    fmha_quant_mode = common.FMHAQuantMode.float16

    context_s_list = [128, 256, 512, 1024, 2048, 4096, 8192, 16384,65536]
    generation_s_list = [128, 512, 1024, 2048, 4096, 8192, 16384, 32768,65536*4]

    fig, ax = plt.subplots(2, 2, figsize=(14, 8))

    context_sol_math_list = []
    context_sol_mem_list = []
    for s in context_s_list:
        sol_time, sol_math, sol_mem = database.query_context_dsa_module(
            b=b,
            s=s,
            prefix=0,
            num_heads=num_heads,
            index_n_heads=index_n_heads,
            index_head_dim=index_head_dim,
            index_topk=index_topk,
            kvcache_quant_mode=kv_cache_dtype,
            fmha_quant_mode=fmha_quant_mode,
            database_mode=DatabaseMode.SOL_FULL,
        )
        db_time = database.query_context_dsa_module(
            b=b,
            s=s,
            prefix=0,
            num_heads=num_heads,
            index_n_heads=index_n_heads,
            index_head_dim=index_head_dim,
            index_topk=index_topk,
            kvcache_quant_mode=kv_cache_dtype,
            fmha_quant_mode=fmha_quant_mode,
            database_mode=DatabaseMode.SILICON,
        )
        context_sol_math_list.append(sol_math / db_time)
        context_sol_mem_list.append(sol_mem / db_time)

    ax[0, 0].plot(context_s_list, context_sol_math_list, color="blue", label="context math")
    ax[0, 1].plot(context_s_list, context_sol_mem_list, color="orange", linestyle="--", label="context mem")
    ax[0, 0].set_title("DSA Context Module: math sol %")
    ax[0, 1].set_title("DSA Context Module: mem sol %")
    ax[0, 0].set_xlabel("s")
    ax[0, 1].set_xlabel("s")
    ax[0, 0].set_ylabel("ratio")
    ax[0, 1].set_ylabel("ratio")
    ax[0, 0].set_ylim(0, 1)
    ax[0, 1].set_ylim(0, 1)
    ax[0, 0].legend()
    ax[0, 1].legend()

    generation_sol_math_list = []
    generation_sol_mem_list = []
    for s in generation_s_list:
        sol_time, sol_math, sol_mem = database.query_generation_dsa_module(
            b=b,
            s=s,
            num_heads=num_heads,
            index_n_heads=index_n_heads,
            index_head_dim=index_head_dim,
            index_topk=index_topk,
            kv_cache_dtype=kv_cache_dtype,
            database_mode=DatabaseMode.SOL_FULL,
        )
        db_time = database.query_generation_dsa_module(
            b=b,
            s=s,
            num_heads=num_heads,
            index_n_heads=index_n_heads,
            index_head_dim=index_head_dim,
            index_topk=index_topk,
            kv_cache_dtype=kv_cache_dtype,
            database_mode=DatabaseMode.SILICON,
        )
        generation_sol_math_list.append(sol_math / db_time)
        generation_sol_mem_list.append(sol_mem / db_time)

    ax[1, 0].plot(generation_s_list, generation_sol_math_list, color="green", label="generation math")
    ax[1, 1].plot(
        generation_s_list,
        generation_sol_mem_list,
        color="red",
        linestyle="--",
        label="generation mem",
    )
    ax[1, 0].set_title("DSA Generation Module: math sol %")
    ax[1, 1].set_title("DSA Generation Module: mem sol %")
    ax[1, 0].set_xlabel("kv cache len (s)")
    ax[1, 1].set_xlabel("kv cache len (s)")
    ax[1, 0].set_ylabel("ratio")
    ax[1, 1].set_ylabel("ratio")
    ax[1, 0].set_ylim(0, 1)
    ax[1, 1].set_ylim(0, 1)
    ax[1, 0].legend()
    ax[1, 1].legend()

    plt.tight_layout()
    plt.show()


#visualize_dsa_module(database)

In [ ]:
def visualize_moe(database):
    workload_distributions = list(list(database._moe_data.values())[0].keys())
    tp_list = [1, 2, 4, 8]
    ep_list = [1, 2, 4, 8, 16, 32]
    tp_ep_list = []
    for tp in tp_list:
        for ep in ep_list:
            if database.backend == "vllm" and tp > 1 and ep > 1:
                continue
            if database.backend == "sglang" and ep > 1:
                continue
            if tp * ep >= 4 and tp * ep <= 32:
                tp_ep_list.append([tp, ep])
    m_list = [1, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    fig, ax = plt.subplots(2*len(workload_distributions), len(tp_ep_list), figsize=(5 * len(tp_ep_list), 5 * 2*len(workload_distributions)))
    for workload_distribution_id, workload_distribution in enumerate(workload_distributions):
        for i, (tp, ep) in enumerate(tp_ep_list):
            for color_id, quant_mode in enumerate(database._moe_data.keys()):
                if quant_mode == common.MoEQuantMode.w4a16_mxfp4:
                    topk = 4
                    num_experts = 128
                    hidden_size = 2880
                    inter_size = 2880
                else:
                    topk = 8
                    num_experts = 256
                    hidden_size = 7168
                    inter_size = 2048

                weight_bits = int(quant_mode.value.memory * 8)
                if (inter_size // tp) % (256 // weight_bits) != 0:
                    continue

                sol_math_list = []
                sol_mem_list = []
                for m in m_list:
                    sol_time, sol_math, sol_mem = database.query_moe(
                        num_tokens=m,
                        hidden_size=hidden_size,
                        inter_size=inter_size,
                        topk=topk,
                        num_experts=num_experts,
                        moe_tp_size=tp,
                        moe_ep_size=ep,
                        quant_mode=quant_mode,
                        workload_distribution=workload_distribution,
                        database_mode=DatabaseMode.SOL_FULL,
                    )
                    db_time = database.query_moe(
                        num_tokens=m,
                        hidden_size=hidden_size,
                        inter_size=inter_size,
                        topk=topk,
                        num_experts=num_experts,
                        moe_tp_size=tp,
                        moe_ep_size=ep,
                        quant_mode=quant_mode,
                        workload_distribution=workload_distribution,
                        database_mode=DatabaseMode.SILICON,
                    )
                    percentage_of_math = sol_math / db_time
                    percentage_of_mem = sol_mem / db_time
                    sol_math_list.append(percentage_of_math)
                    sol_mem_list.append(percentage_of_mem)

                ax[workload_distribution_id*2, i].plot(
                    m_list, sol_math_list, color=color_list[color_id], label=f"{quant_mode} math"
                )
                ax[workload_distribution_id*2+1, i].plot(
                    m_list,
                    sol_mem_list,
                    color=color_list[color_id],
                    linestyle="--",
                    label=f"{quant_mode} mem",
                )
            if workload_distribution != "balanced":
                workload_distribution_title = workload_distribution + "  vs balanced sol"
            else:
                workload_distribution_title = workload_distribution

            ax[workload_distribution_id*2, i].set_title(f"{workload_distribution_title} \ntopk={topk} e={num_experts} tp={tp} ep={ep}")
            ax[workload_distribution_id*2, i].set_xlabel("s")
            ax[workload_distribution_id*2, i].set_ylabel("math sol %")
            # ax[0,i].set_ylim(0,1)
            ax[workload_distribution_id*2, i].legend()
            ax[workload_distribution_id*2+1, i].set_xlabel("s")
            ax[workload_distribution_id*2+1, i].set_ylabel("mem sol %")
            # ax[1,i].set_ylim(0,1)
            ax[workload_distribution_id*2+1, i].legend()
    plt.tight_layout()
    plt.show()


visualize_moe(database)

In [ ]:
def visualize_allreduce(database):
    quant_mode = common.CommQuantMode.half
    m_list = [
        2**0,
        2**1,
        2**2,
        2**4,
        2**6,
        2**8,
        2**10,
        2**12,
        2**14,
        2**16,
        2**18,
        2**20,
        2**22,
        2**24,
        2**26,
        2**28,
        2**30,
        2**32,
    ]
    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    if database.system in ["gb200", "gb300"]:
        tp_list = [2, 4]
    else:
        tp_list = [2, 4, 8, 16, 32]
    fig, ax = plt.subplots(1, len(tp_list), figsize=(5 * len(tp_list), 5 * 1))
    for i, tp_size in enumerate(tp_list):
        sol_list = []
        for m in m_list:
            sol_time, sol_math, sol_mem = database.query_custom_allreduce(
                quant_mode, tp_size, m, database_mode=DatabaseMode.SOL_FULL
            )
            db_time = database.query_custom_allreduce(quant_mode, tp_size, m, database_mode=DatabaseMode.SILICON)
            percentage_of_sol = sol_time / db_time
            sol_list.append(percentage_of_sol)
        ax[i].plot(m_list, sol_list, color=color_list[i], label=f"{tp_size}")
        ax[i].set_title(f"tp {tp_size}")
        # ax[i].set_xscale('log', base=2)
        ax[i].set_xlabel("message_size")
        ax[i].set_ylabel("sol %")
        # ax[i].set_ylim(0,1)
        ax[i].legend()
    plt.tight_layout()
    plt.show()


visualize_allreduce(database)

In [ ]:
def visualize_nccl(database, operation="all_gather"):
    quant_mode = common.CommQuantMode.half
    m_list = [
        2**0,
        2**1,
        2**2,
        2**4,
        2**6,
        2**8,
        2**10,
        2**12,
        2**14,
        2**16,
        2**18,
        2**20,
        2**22,
        2**24,
        2**26,
        2**28,
        2**30,
        2**32,
    ]
    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]
    num_gpu_list = [2, 4, 8, 16, 32, 64]
    fig, ax = plt.subplots(1, len(num_gpu_list), figsize=(5 * len(num_gpu_list), 5 * 1))
    for i, num_gpu in enumerate(num_gpu_list):
        sol_list = []
        for m in m_list:
            sol_time, sol_math, sol_mem = database.query_nccl(
                quant_mode, num_gpu, operation, m, database_mode=DatabaseMode.SOL_FULL
            )
            db_time = database.query_nccl(
                quant_mode, num_gpu, operation, m, database_mode=DatabaseMode.SILICON
            )
            percentage_of_sol = sol_time / db_time
            sol_list.append(percentage_of_sol)
        ax[i].plot(m_list, sol_list, color=color_list[i], label=f"{num_gpu}")
        ax[i].set_title(f"{operation} num_gpu {num_gpu}")
        # ax[i].set_xscale('log', base=2)
        ax[i].set_xlabel("message_size")
        ax[i].set_ylabel("sol %")
        # ax[i].set_ylim(0,1)
        ax[i].legend()
    plt.tight_layout()
    plt.show()


op_list = ["all_gather", "all_reduce", "alltoall", "reduce_scatter"]
for op in op_list:
    visualize_nccl(database, operation=op)

In [ ]:
def visualize_trtllm_alltoall(database):
    """Visualize TRT-LLM AlltoAll communication latency and sol%.

    Layout: rows = op phases (prepare/dispatch/combine/combine_lp) with latency and sol%,
            cols = ep_sizes, lines = moe_dtype (quant mode).

    """
    alltoall_data = database._trtllm_alltoall_data
    if not alltoall_data.loaded:
        print("No trtllm alltoall data available (data not loaded)")
        return
    if not alltoall_data:
        print("No trtllm alltoall data available (empty)")
        return

    color_list = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "olive",
        "cyan",
    ]

    # Map kernel_source -> moe_backend for query API
    kernel_to_backend = {
        "NVLinkTwoSided": "wideep",
        "NVLinkOneSided": None,
    }

    for kernel_source in alltoall_data:
        if kernel_source not in kernel_to_backend:
            continue
        moe_backend = kernel_to_backend[kernel_source]
        kernel_data = alltoall_data[kernel_source]

        op_names_set = set()
        ep_sizes_set = set()
        quant_modes_set = set()

        for op_name in kernel_data:
            op_names_set.add(op_name)
            for qm in kernel_data[op_name]:
                quant_modes_set.add(qm)
                for nn in kernel_data[op_name][qm]:
                    for hs in kernel_data[op_name][qm][nn]:
                        for tk in kernel_data[op_name][qm][nn][hs]:
                            for ne in kernel_data[op_name][qm][nn][hs][tk]:
                                for ep in kernel_data[op_name][qm][nn][hs][tk][ne]:
                                    ep_sizes_set.add(ep)

        op_order = [
            "alltoall_prepare",
            "alltoall_dispatch",
            "alltoall_combine",
            "alltoall_combine_low_precision",
        ]
        op_names = [op for op in op_order if op in op_names_set]
        ep_sizes = sorted(ep_sizes_set)
        quant_modes = sorted(quant_modes_set, key=str)

        if not op_names or not ep_sizes:
            continue

        sol_ops = {"alltoall_dispatch", "alltoall_combine", "alltoall_combine_low_precision"}
        rows = []
        for op in op_names:
            rows.append((op, "latency"))
            if op in sol_ops:
                rows.append((op, "sol %"))

        n_rows = len(rows)
        n_cols = max(len(ep_sizes), 1)
        fig, axes = plt.subplots(
            n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False,
        )
        fig.suptitle(
            f"{database.system.upper()} - {database.backend.upper()} {database.version}"
            f" - TRT-LLM AlltoAll Sanity Chart ({kernel_source})",
            fontsize=14,
            fontweight="bold",
        )

        for ri, (op_name, metric) in enumerate(rows):
            for ci, target_ep in enumerate(ep_sizes):
                ax = axes[ri][ci]
                cid = 0
                for qm in quant_modes:
                    # Discover available num_tokens for this (qm, op, ep) combo
                    token_set, hs, tk, ne_val, nn_val = set(), 0, 0, 0, 0
                    if qm in kernel_data.get(op_name, {}):
                        for nn in kernel_data[op_name][qm]:
                            for h in kernel_data[op_name][qm][nn]:
                                for t in kernel_data[op_name][qm][nn][h]:
                                    for ne in kernel_data[op_name][qm][nn][h][t]:
                                        if target_ep not in kernel_data[op_name][qm][nn][h][t][ne]:
                                            continue
                                        hs, tk, ne_val, nn_val = h, t, ne, nn
                                        token_set.update(kernel_data[op_name][qm][nn][h][t][ne][target_ep].keys())

                    if not token_set:
                        cid += 1
                        continue

                    tokens = sorted(token_set)
                    label = qm.name if hasattr(qm, "name") else str(qm)

                    query_kwargs = dict(
                        op_name=op_name, hidden_size=hs, topk=tk,
                        num_experts=ne_val, moe_ep_size=target_ep,
                        quant_mode=qm, node_num=nn_val, moe_backend=moe_backend,
                    )

                    if metric == "latency":
                        vals = [
                            float(database.query_trtllm_alltoall(
                                num_tokens=nt, database_mode=DatabaseMode.SILICON, **query_kwargs,
                            ))
                            for nt in tokens
                        ]
                    else:
                        vals = []
                        for nt in tokens:
                            sol_time = database.query_trtllm_alltoall(
                                num_tokens=nt, database_mode=DatabaseMode.SOL_FULL, **query_kwargs,
                            )[0]
                            db_time = database.query_trtllm_alltoall(
                                num_tokens=nt, database_mode=DatabaseMode.SILICON, **query_kwargs,
                            )
                            vals.append(sol_time / db_time if db_time > 0 else 0)

                    ax.plot(
                        tokens,
                        vals,
                        color=color_list[cid % len(color_list)],
                        label=label,
                        marker=".",
                        markersize=3,
                    )
                    cid += 1

                ax.set_title(f"{op_name}\nep_size={target_ep}")
                ax.set_xlabel("num_tokens")
                ax.set_ylabel(metric)
                ax.legend(fontsize="small")

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()


visualize_trtllm_alltoall(database)